## Plot correlation between following errors and elevation angle

##### **Description**


Struts 5 and 6 in the M2 Hexapod support most of the M2 weight when the telescope is at the horizon position [(see Figure 2 on this page)](https://ts-mthexapod.lsst.io/algorithm/kinematics.html). This load might add a correlation between the faults and the elevation angle. 

We want a histogram showing the number of Following Error faults per elevation angle. We can bin the elevation angles for every 5º. 

LSSTCam data can be analysed. ComCam data too, however with some missing data for some dates.
The challenge here will be to build a query to the EFD that can query the faults without downloading too much data. 

##### **What does this Notebook do?**

- It first finds the error code for the faults on M2 hexapod.
- Then creates the corresponding timestamps
- Plots the corresponding elevation angle (one second before it goes to fault)
- Creates a histogram of the number of faults per elevation angle
- In the end, it plots the strut lengths (all 6 struts) just before M2 hexapod goes to fault


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
    
from datetime import timedelta
from matplotlib import pyplot as plt

from astropy.time import Time
import numpy as np
from lsst.summit.utils.efdUtils import (
    getEfdData,
    makeEfdClient,
)    

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()


def query_mtmount_elevation_telemetry(client, t_stamp, delta_t=1):
    """Query the MTMount elevation telemetry for a given `t_stamp`"""
    t_stamp = Time(t_stamp)
    start_time = t_stamp - timedelta(seconds=delta_t)
    end_time = t_stamp

    _df = getEfdData(
        client=client,
        topic="lsst.sal.MTMount.elevation",
        columns="actualPosition",
        begin=start_time,
        end=end_time,
    )

    return _df

## Data Analysis

Retrieving the `errorCode` and corresponding `errorReport` for a given `Fault` event Timestamp.\
The `salIndex` can be either 1 (Camera Hexapod) or 2 (M2 Hexapod).
- LSSTCam data from April 20, 2025.
- ComCam data between October 24 2024 and December 10 2024 (data missing for November 1 and December 1 2024, even if there are M2 Hexapod faults).

The meaning of the error codes can be found here : [Error codes](https://github.com/lsst-ts/ts_xml/blob/1a99e1e294d1464eb9a4208b531540dae95cd1fb/python/lsst/ts/xml/enums/MTHexapod.py#L74)

errorCode =2 is related to loss of connection, which seems to impact elevation angle data logging.\
We want to disard this kind of error.

In [ ]:
#######
# ComCam
# cc_start1 = Time("2024-10-24T00:00:00", scale="utc", format="isot")
# cc_end1 = Time("2025-11-01T00:00:00", scale="utc", format="isot")
#######
# LSSTCam
cc_start1 = Time("2025-04-20T8:00:00", scale="utc", format="isot")
cc_end1 = Time("2025-08-25T10:00:00", scale="utc", format="isot")

In [ ]:
# get the error events when system goes to fault
#salIndex = 2 is for M2 Hexapod
# we discard errorCode = 0 => it's when the previous error is lifted
# and we discard errorCode = 2 caused by connection problems which cause data loss
query = f"""
     SELECT
     errorCode, salIndex, errorReport
     FROM "lsst.sal.MTHexapod.logevent_errorCode"
     WHERE time >= '{cc_start1.isot}Z'
     AND time <= '{cc_end1.isot}Z'
     AND salIndex = 2
     AND errorCode != 0
     AND errorCode != 2 
"""

mthexerrcode_df = await efd_client.influx_client.query(query)

In [ ]:
# Create timestamps related to fault events 
mthexerrcode_t = np.array([])
for i in mthexerrcode_df.index:
    mthexerrcode_t = np.append(mthexerrcode_t, [i])

In [ ]:
# Get elevation angle corresponding to the Fault timestamps
hexeleva = []
for i in range(len(mthexerrcode_t)):
    hexelev = query_mtmount_elevation_telemetry(efd_client, mthexerrcode_t[i], delta_t=1)
    if len(hexelev) != 0:
        hexeleva.append(hexelev["actualPosition"].iloc[-1])

Make sure the timestamps (mthexerrcode_t) and the elevation (hexeleva) arrays are of same dimensions.\
Sometimes there's missing data in elevation (Noticed this in ComCam data)

In [ ]:
# plot elevation vs fault times
fig, ax = plt.subplots(1, 1, dpi=125, figsize=(8, 4))
ax.plot(mthexerrcode_t, hexeleva, marker="+", linestyle="")
ax.set(
    ylabel="Elevation angle (degrees)",
    xlabel="Date",
    title="Elevation angle just before M2Hex faults; LSSTCam",
)
plt.xticks(rotation=90)
fig.tight_layout()

In [ ]:
# histogram
fig, ax = plt.subplots(1, 1, dpi=125, figsize=(8, 4))
binwidth = 8
ax.hist(hexeleva, bins=range(0, 100 + 10, 5))
ax.set(
    ylabel="Fault Counts",
    xlabel="Elevation angle (degrees)",
    title="Counts TMA is at a given elevation angle just before M2Hex faults; LSSTCam",
)
fig.tight_layout()

## Comments

- It looks like the M2 Hexapod faults more often at higher elevation, however, the TMA is more often at higher elevations than at lower elevation.
It might be interesting to look at the ratio of faults at a given angle with respect to the number of times TMA is at that elevation angle.
However, this requires other ways of accessing the data. => subject of another study...TB
- If you are interested in looking at the actuator positions when the M2 Hexapod goes to Fault: subject of another study [SITCOM-2204](https://rubinobs.atlassian.net/browse/SITCOM-2204)